# 外汇汇率预测 Transformer 模型

本笔记演示了如何使用 PyTorch 构建 Transformer 模型，对 Excel 中的外汇汇率特征进行建模、训练与预测。

## 使用说明
1. 在下方的 **Configuration** 区块中，根据自己的 Excel 文件路径及字段，指定自变量（特征列）与因变量（目标列）。
2. 确保 Excel 文件中包含 `Date` 日期列，或在配置中指定正确的日期列名称。
3. 运行每个代码单元格以完成数据加载、特征工程、模型训练、评估与可视化。

In [ ]:

# Configuration
from pathlib import Path
CONFIG = {
    "excel_path": Path("data/forex_features.xlsx"),  # Excel 文件路径
    "date_column": "Date",  # 日期列名称
    "feature_columns": [
        # 在此列出作为自变量的列名，例如:
        # "USD_Index", "CPI", "Interest_Rate"
    ],
    "target_column": "Target",  # 因变量列名称（需要预测的汇率）
    "test_ratio": 0.2,  # 测试集比例
    "lookback": 30,  # 序列窗口长度
    "horizon": 1,  # 预测步长（1 表示预测下一期）
    "batch_size": 32,
    "epochs": 100,
    "learning_rate": 1e-3,
    "d_model": 64,
    "nhead": 8,
    "num_layers": 2,
    "dim_feedforward": 128,
    "dropout": 0.1,
    "seed": 42,
    "device": "cuda"  # 如果有 GPU，可设置为 "cuda"，否则保持 "cpu"
}


In [ ]:

# Imports
import math
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
device = torch.device(CONFIG["device"] if torch.cuda.is_available() and CONFIG["device"] == "cuda" else "cpu")
print(f"Using device: {device}")


In [ ]:

# Data Loading
excel_path = Path(CONFIG["excel_path"])
if not excel_path.exists():
    raise FileNotFoundError(f"未找到 Excel 文件: {excel_path.resolve()}")

date_col = CONFIG["date_column"]
feature_cols = CONFIG["feature_columns"]
target_col = CONFIG["target_column"]

if not feature_cols:
    raise ValueError("请在 CONFIG['feature_columns'] 中至少指定一个自变量列名。")

df = pd.read_excel(excel_path, parse_dates=[date_col])
if df[date_col].isna().any():
    raise ValueError("日期列包含缺失值，请清理数据后再运行。")

df = df.drop_duplicates(subset=[date_col]).set_index(date_col).sort_index()

missing_cols = [col for col in feature_cols + [target_col] if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下列在数据中不存在: {missing_cols}")

display(df.head())


In [ ]:

# Descriptive Statistics
display(df[feature_cols + [target_col]].describe().T)
display(pd.DataFrame({"missing_values": df[feature_cols + [target_col]].isna().sum()}))


In [ ]:

# Train/Test Split and Scaling
test_ratio = CONFIG["test_ratio"]
lookback = CONFIG["lookback"]
horizon = CONFIG["horizon"]

if not 0 < test_ratio < 1:
    raise ValueError("test_ratio 必须在 0 与 1 之间。")

if lookback <= 0:
    raise ValueError("lookback 必须为正整数。")

if horizon <= 0:
    raise ValueError("horizon 必须为正整数。")

total_len = len(df)
test_size = max(int(total_len * test_ratio), 1)
train_df = df.iloc[: total_len - test_size]
test_df = df.iloc[total_len - test_size - lookback :]

feature_scaler = StandardScaler().fit(train_df[feature_cols])
target_scaler = StandardScaler().fit(train_df[[target_col]])

def create_sequences(dataframe):
    features = feature_scaler.transform(dataframe[feature_cols]).astype(np.float32)
    targets = target_scaler.transform(dataframe[[target_col]]).astype(np.float32)

    seq_features, seq_targets = [], []
    for idx in range(lookback, len(dataframe) - horizon + 1):
        seq_features.append(features[idx - lookback : idx])
        seq_targets.append(targets[idx + horizon - 1])

    if not seq_features:
        raise ValueError("给定的 lookback/horizon 设置无法生成任何序列，请调整参数。")

    return (
        torch.tensor(np.stack(seq_features)),
        torch.tensor(np.stack(seq_targets)).squeeze(-1),
    )

train_features, train_targets = create_sequences(train_df)
test_features, test_targets = create_sequences(test_df)

class SequenceDataset(Dataset):
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

train_dataset = SequenceDataset(train_features, train_targets)
test_dataset = SequenceDataset(test_features, test_targets)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, drop_last=False)

print(f"Train sequences: {len(train_dataset)} | Test sequences: {len(test_dataset)}")


In [ ]:

# Model Definition
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, dim_feedforward, dropout, output_size=1):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.positional_encoding = PositionalEncoding(d_model=d_model, dropout=dropout)
        self.regressor = nn.Linear(d_model, output_size)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        output = self.regressor(x[:, -1, :])
        return output.squeeze(-1)


model = TimeSeriesTransformer(
    input_size=len(CONFIG["feature_columns"]),
    d_model=CONFIG["d_model"],
    nhead=CONFIG["nhead"],
    num_layers=CONFIG["num_layers"],
    dim_feedforward=CONFIG["dim_feedforward"],
    dropout=CONFIG["dropout"],
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])


In [ ]:

# Training Loop
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    train_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            val_loss += loss.item() * batch_x.size(0)

    val_loss /= len(test_loader.dataset)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if epoch % max(1, CONFIG["epochs"] // 10) == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


In [ ]:

# Evaluation on Test Set
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        preds = model(batch_x).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(batch_y.cpu().numpy())

preds_scaled = np.concatenate(all_preds)
targets_scaled = np.concatenate(all_targets)

preds = target_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
targets = target_scaler.inverse_transform(targets_scaled.reshape(-1, 1)).flatten()

mae = mean_absolute_error(targets, preds)
rmse = mean_squared_error(targets, preds, squared=False)
r2 = r2_score(targets, preds)

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R^2 : {r2:.4f}")

history_df = pd.DataFrame(history)
ax = history_df.plot(title="Training & Validation Loss", figsize=(8, 4))
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
plt.show()


In [ ]:

# Prediction Plot
test_indices = test_df.index[lookback - 1 : lookback - 1 + len(preds)]
result_df = pd.DataFrame({
    "Actual": targets,
    "Predicted": preds,
}, index=test_indices)

ax = result_df.plot(figsize=(12, 5), title="Actual vs. Predicted Exchange Rate")
ax.set_xlabel("Date")
ax.set_ylabel(target_col)
plt.show()

display(result_df.head())
